In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import re
import os
import math

print('Libraries imported!')

Libraries imported!



## 1. Project Configuration


| Constant | Purpose |
|---|---|
| `BASE_DIR` | Root of the project (notebook's working directory) |
| `RAW_DIR` | `BASE_DIR/data/raw` — raw OHLCV CSVs |
| `PROCESSED_DIR` | `BASE_DIR/data/processed` — feature-engineered CSVs |

In [18]:
BASE_DIR      = Path("/home/rishabhmeel/DAI_project")
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

print('RAW_DIR      :', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

RAW_DIR      : /home/rishabhmeel/DAI_project/data/raw
PROCESSED_DIR: /home/rishabhmeel/DAI_project/data/processed


# 2. Loading the data

In [19]:
df = pd.read_excel(os.path.join(RAW_DIR,'datasheet.xls'))
df
# for col in df.columns:
#     print(repr(col))

,District Names,State/UT,Number of Households surveyed,Number of Women age 15-49 years interviewed,Number of Men age 15-54 years interviewed,Female population age 6 years and above who ever attended school (%),Population below age 15 years (%),"Sex ratio of the total population (females per 1,000 males)","Sex ratio at birth for children born in the last five years (females per 1,000 males)",Children under age 5 years whose birth was registered with the civil authority (%),...,Men age 15 years and above wih Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%),Men age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%),Men age 15 years and above wih Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%),Women (age 30-49 years) Ever undergone a screening test for cervical cancer (%),Women (age 30-49 years) Ever undergone a breast examination for breast cancer (%),Women (age 30-49 years) Ever undergone an oral cavity examination for oral cancer (%),Women age 15 years and above who use any kind of tobacco (%),Men age 15 years and above who use any kind of tobacco (%),Women age 15 years and above who consume alcohol (%),Men age 15 years and above who consume alcohol (%)
0,Nicobars,Andaman & Nicobar Islands,882,764,125,78.01,22.98,973.31,927.41,98.01,...,32.86,11.06,46.97,13.35,13.16,5.37,63.46,76.79,29.59,64.49
1,North & Middle Andaman,Andaman & Nicobar Islands,874,789,108,82.66,19.82,949.82,844.43,100.00,...,22.62,5.97,32.20,1.70,0.25,15.84,46.77,70.47,5.08,45.26
2,South Andaman,Andaman & Nicobar Islands,868,844,134,84.68,20.95,967.48,934.92,96.53,...,17.88,6.09,26.90,1.32,0.67,8.00,19.60,50.76,1.72,32.81
3,Srikakulam,Andhra Pradesh,874,780,100,59.99,20.71,1139.51,1162.58,94.96,...,14.37,5.51,22.89,1.04,0.23,3.76,7.09,21.32,0.55,28.26
4,Vizianagaram,Andhra Pradesh,902,853,134,55.95,20.56,1114.35,898.03,95.42,...,14.83,6.43,25.13,4.90,0.63,7.33,11.41,21.47,0.80,32.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
702,South Twenty Four Pargana,West Bengal,917,1089,148,80.93,25.49,1054.81,1099.04,97.56,...,13.03,4.28,19.13,0.00,0.00,0.20,6.28,49.76,0.39,16.54
703,Paschim Medinipur,West Bengal,910,1002,137,72.10,23.43,1088.57,1005.04,99.32,...,10.66,4.33,18.98,0.00,0.00,0.00,18.83,50.48,1.46,20.81
704,Purba Medinipur,West Bengal,873,957,131,83.04,23.85,1031.40,777.38,99.25,...,11.92,4.40,21.31,0.00,0.19,0.46,12.93,40.11,1.97,14.80
705,Paschim Barddhaman,West Bengal,912,1190,184,74.84,22.32,977.46,786.93,97.70,...,14.45,3.67,20.67,0.70,1.30,0.40,11.79,40.23,0.83,14.58


# 3. Choosing the particular columns

In [20]:
cols_to_keep = [
    "District Names",
    "State/UT",
    "Population living in households that use an improved sanitation facility2 (%)",
    "Women age 15-24 years who use hygienic methods of protection during their menstrual period5 (%)",
    "Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%)",
    "Total children age 6-23 months receiving an adequate diet16, 17  (%)",
    "All women age 15-49 years who are anaemic22 (%)",
    "Men age 15 years and above wih high or very high (>140 mg/dl) Blood sugar level  or taking medicine to control blood sugar level23 (%)",
    "Women age 15 years and above who use any kind of tobacco (%)",
    "Men age 15 years and above who consume alcohol (%)",
]

df = df[cols_to_keep].copy()
display(df)

,District Names,State/UT,Population living in households that use an improved sanitation facility2 (%),Women age 15-24 years who use hygienic methods of protection during their menstrual period5 (%),Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%),"Total children age 6-23 months receiving an adequate diet16, 17 (%)",All women age 15-49 years who are anaemic22 (%),Men age 15 years and above wih high or very high (>140 mg/dl) Blood sugar level or taking medicine to control blood sugar level23 (%),Women age 15 years and above who use any kind of tobacco (%),Men age 15 years and above who consume alcohol (%)
0,Nicobars,Andaman & Nicobar Islands,83.45,100.00,-94.08,-18.71,38.30,15.39,63.46,64.49
1,North & Middle Andaman,Andaman & Nicobar Islands,86.38,100.00,*,-5.92,62.07,18.28,46.77,45.26
2,South Andaman,Andaman & Nicobar Islands,89.33,98.19,-96.57,23.49,57.66,18.10,19.60,32.81
3,Srikakulam,Andhra Pradesh,71.58,77.95,-100,16.14,62.59,17.57,7.09,28.26
4,Vizianagaram,Andhra Pradesh,61.74,72.52,*,1.75,64.04,14.47,11.41,32.30
...,...,...,...,...,...,...,...,...,...,...
702,South Twenty Four Pargana,West Bengal,70.08,80.31,96.34,27.55,61.62,21.28,6.28,16.54
703,Paschim Medinipur,West Bengal,58.33,67.31,86.82,21.78,81.49,21.67,18.83,20.81
704,Purba Medinipur,West Bengal,69.78,78.42,-74.29,20.93,70.39,24.04,12.93,14.80
705,Paschim Barddhaman,West Bengal,71.46,84.42,69.69,31.05,65.81,23.40,11.79,14.58


# 4. Adding the MPI with the main data

In [21]:
df_mpi = pd.read_excel(os.path.join(RAW_DIR, 'india_district_mpi_nfhs4_nfhs5.xlsx'))

# Tidy the headers and drop empty rows
df_mpi.columns = df_mpi.columns.astype(str).str.strip()
df_mpi = df_mpi.dropna(how='all').reset_index(drop=True)

print(df_mpi.shape)
print(df_mpi.columns.tolist())
display(df_mpi)

(709, 8)
['District', 'State', 'NFHS-4 (2015-16) Headcount Ratio', 'NFHS-4 (2015-16) Intensity', 'NFHS-4 (2015-16) MPI', 'NFHS-5 (2019-21) Headcount Ratio', 'NFHS-5 (2019-21) Intensity', 'NFHS-5 (2019-21) MPI']


,District,State,NFHS-4 (2015-16) Headcount Ratio,NFHS-4 (2015-16) Intensity,NFHS-4 (2015-16) MPI,NFHS-5 (2019-21) Headcount Ratio,NFHS-5 (2019-21) Intensity,NFHS-5 (2019-21) MPI
0,Nicobars,Andaman & Nicobar Islands,0.0528,0.3680,0.019,0.0340,0.3942,0.013
1,North & Middle Andaman,Andaman & Nicobar Islands,0.0932,0.4182,0.039,0.0439,0.4251,0.019
2,South Andamans,Andaman & Nicobar Islands,0.0220,0.3923,0.009,0.0115,0.3766,0.004
3,Anantapur,Andhra Pradesh,0.1247,0.4200,0.052,0.0674,0.4056,0.027
4,Chittoor,Andhra Pradesh,0.0964,0.4265,0.041,0.0566,0.3920,0.022
...,...,...,...,...,...,...,...,...
704,Purba Bardhaman,West Bengal,NaN,NaN,NaN,0.1432,0.4118,0.059
705,Purba Medinipur (Medinipur East),West Bengal,0.1419,0.4268,0.061,0.1248,0.4116,0.051
706,Puruliya,West Bengal,0.4969,0.4744,0.236,0.2684,0.4374,0.117
707,South 24 Parganas,West Bengal,0.2810,0.4571,0.128,0.1096,0.4122,0.045


In [22]:
import difflib

def norm(s):
    return (s.astype(str).str.strip().str.lower()
              .str.replace('&', 'and', regex=False)
              .str.replace(r'[^a-z0-9 ]', '', regex=True)
              .str.replace(r'\s+', ' ', regex=True).str.strip())

def find_col(d, *cands):
    sq = lambda x: re.sub(r'[^a-z0-9]', '', str(x).lower())
    lk = {sq(c): c for c in d.columns}
    for c in cands:
        if sq(c) in lk: return lk[sq(c)]
    raise KeyError(f'{cands} not in {d.columns.tolist()}')

df_state,  df_dist  = find_col(df, 'State/UT', 'state_ut', 'state'), find_col(df, 'District Names', 'district_names', 'district')
mpi_state, mpi_dist = find_col(df_mpi, 'state', 'State/UT'), find_col(df_mpi, 'district', 'District Names')

# State spellings that differ between the two files
STATE_ALIAS = {'maharastra': 'maharashtra', 'nct of delhi': 'delhi'}

# Same district, different name (renamed / re-labelled). Keys are normalised (state, df name) -> mpi name
MANUAL = {
    ('sikkim', 'east district'):  'gangtok',
    ('sikkim', 'west district'):  'gyalshing',
    ('sikkim', 'north district'): 'mangan',
    ('sikkim', 'south district'): 'namchi',
    ('uttar pradesh', 'kanshiram nagar'): 'kasganj',
    ('mizoram', 'chandel'): 'saiha',              # datasheet row is labelled Chandel/Mizoram but holds Saiha
    ('andhra pradesh', 'sri potti sriramulu nello'): 'spsr nellore',
    ('chhattisgarh', 'koriya'): 'korea',
    ('gujarat', 'the dangs'): 'dang',
    ('haryana', 'gurgaon'): 'gurugram',
    ('karnataka', 'belgaum'): 'belagavi',
    ('karnataka', 'mysore'): 'mysuru',
    ('odisha', 'subarnapur'): 'sonepur',
    ('west bengal', 'haora'): 'howrah',
    ('west bengal', 'north twenty four pargana'): 'north 24 parganas',
    ('west bengal', 'south twenty four pargana'): 'south 24 parganas',
}

def name_keys(name):
    """'Bid (Beed)' -> {'bid beed', 'bid', 'beed'}: lets us match names with a bracketed alias."""
    name = str(name)
    keys = {norm(pd.Series([re.sub(r'[()]', ' ', name)]))[0]}
    m = re.match(r'^(.*?)\((.*)\)\s*$', name)
    if m:
        keys |= {norm(pd.Series([m.group(1)]))[0], norm(pd.Series([m.group(2)]))[0]}
    return keys

left, right = df.copy().reset_index(drop=True), df_mpi.copy().reset_index(drop=True)
left['_lid'],  right['_rid']  = left.index, right.index

# Health indicators: '*' (suppressed) -> NaN, and values the datasheet stores as negatives
# (its way of showing "(xx.x)" = based on 25-49 cases) -> positive, shown to 1 decimal like the factsheet
health_cols = [c for c in df.columns if c not in (df_state, df_dist)]
for c in health_cols:
    left[c] = pd.to_numeric(left[c], errors='coerce')
    neg = left[c] < 0
    half_up = np.floor(np.round(left[c].abs() * 100) / 10 + 0.5) / 10      # 85.45 -> 85.5 (Excel style)
    left[c] = left[c].abs().where(~neg, half_up)
left['_state'],  left['_dist']  = norm(left[df_state]).replace(STATE_ALIAS),  norm(left[df_dist])
right['_state'], right['_dist'] = norm(right[mpi_state]).replace(STATE_ALIAS), norm(right[mpi_dist])
left['_keys']  = left[df_dist].map(name_keys)
right['_keys'] = right[mpi_dist].map(name_keys)

L = left[['_lid', '_state', '_dist', '_keys']]
R = right[['_rid', '_state', '_dist', '_keys']]
print('rows -> df:', len(L), '| df_mpi:', len(R))

# Diagnostic: state names that don't exist on both sides (should be empty now)
print('\nStates only in df    :', sorted(set(L._state) - set(R._state)))
print('States only in df_mpi:', sorted(set(R._state) - set(L._state)))

def unique_pairs(a, b, on):
    p = a.merge(b, on=on)[['_lid', '_rid']]
    return p.drop_duplicates('_lid', keep=False).drop_duplicates('_rid', keep=False)

# Stage 1: exact (aliased) state + district
s1 = unique_pairs(L[['_lid', '_state', '_dist']], R[['_rid', '_state', '_dist']], ['_state', '_dist'])
s1['method'] = 'state+district'

# Stage 2: same state, district names match after removing brackets (e.g. 'Bid' ~ 'Bid (Beed)')
L2, R2 = L[~L._lid.isin(s1._lid)], R[~R._rid.isin(s1._rid)]
rows = []
for _, l in L2.iterrows():
    hit = R2[(R2._state == l._state) & R2._keys.map(lambda k: bool(k & l._keys))]
    if len(hit) == 1: rows.append((l._lid, hit._rid.iloc[0], 'state+bracket alias'))
s2 = pd.DataFrame(rows, columns=['_lid', '_rid', 'method'])
s2 = s2.drop_duplicates('_rid', keep=False)

# Stage 3: manual overrides for districts that were renamed / relabelled
L3, R3 = L2[~L2._lid.isin(s2._lid)], R2[~R2._rid.isin(s2._rid)]
rows = []
for _, l in L3.iterrows():
    tgt = MANUAL.get((l._state, l._dist))
    if tgt:
        hit = R3[(R3._state == l._state) & (R3._dist == tgt)]
        if len(hit) == 1: rows.append((l._lid, hit._rid.iloc[0], 'manual'))
s3 = pd.DataFrame(rows, columns=['_lid', '_rid', 'method'])

# Stage 4: fuzzy district match inside the same state (spelling variants)
L4, R4 = L3[~L3._lid.isin(s3._lid)], R3[~R3._rid.isin(s3._rid)]
rows, taken = [], set()
for _, l in L4.iterrows():
    best, best_d = None, 0
    for _, r in R4[R4._state == l._state].iterrows():
        if r['_rid'] in taken: continue
        d = max(difflib.SequenceMatcher(None, a, b).ratio() for a in l._keys for b in r._keys)
        if d >= 0.75 and d > best_d:
            best, best_d = r, d
    if best is not None:
        rows.append((l['_lid'], best['_rid'], 'fuzzy'))
        taken.add(best['_rid'])
s4 = pd.DataFrame(rows, columns=['_lid', '_rid', 'method'])

# Stage 5: district name only (state label wrong in one file, e.g. Chandel), unambiguous names only
L5, R5 = L4[~L4._lid.isin(s4._lid)], R4[~R4._rid.isin(s4._rid)]
s5 = unique_pairs(L5[['_lid', '_dist']], R5[['_rid', '_dist']], ['_dist'])
s5['method'] = 'district only'

pairs = pd.concat([s1, s2, s3, s4, s5], ignore_index=True)
print('\nMatched by method:\n', pairs['method'].value_counts().to_string())
print('Total matched:', len(pairs))

# Review the non-exact matches before trusting them
fz = (pairs[pairs.method != 'state+district']
        .merge(left[['_lid', df_state, df_dist]], on='_lid')
        .merge(right[['_rid', mpi_state, mpi_dist]], on='_rid'))
print('\nNon-exact matches to review:\n', fz[['method', df_state, df_dist, mpi_state, mpi_dist]].to_string())

# Still unmatched on each side
print('\nUnmatched in df:\n',
      left.loc[~left._lid.isin(pairs._lid), [df_state, df_dist]].to_string())
print('\nUnmatched in df_mpi (not in the NFHS-5 datasheet):\n',
      right.loc[~right._rid.isin(pairs._rid), [mpi_state, mpi_dist]].to_string())

# Build the merged dataframe: District, State, NFHS-5 MPI + the 8 health indicators
MPI_COL = find_col(df_mpi, 'NFHS-5 (2019-21) MPI')
merged_df = (pairs
    .merge(left.drop(columns=['_state', '_dist', '_keys']), on='_lid')
    .merge(right[['_rid', mpi_state, mpi_dist, MPI_COL]], on='_rid')
    .sort_values('_rid').reset_index(drop=True))

# Same column layout as MPI_vs_Health_Indicators.xlsx: District, State, MPI, then health columns
merged_df = merged_df[[mpi_dist, mpi_state, MPI_COL] + health_cols]
merged_df.columns = ['District', 'State', MPI_COL] + health_cols

# Tidy headers / names exactly like the expected file: drop brackets, "(%)" becomes "%"
strip_br = lambda s: re.sub(r'\s+', ' ', re.sub(r'[()]', ' ', str(s))).strip()
merged_df.columns = [c if c in ('District', 'State') else re.sub(r'[()]', '', c) for c in merged_df.columns]
merged_df['District'] = merged_df['District'].map(strip_br)
print('\nmerged_df shape:', merged_df.shape)
display(merged_df)

rows -> df: 707 | df_mpi: 709

States only in df    : []
States only in df_mpi: []

Matched by method:
 method
state+district         614
fuzzy                   47
state+bracket alias     30
manual                  16
Total matched: 707

Non-exact matches to review:
                  method                   State/UT               District Names                      State                                  District
0   state+bracket alias             Andhra Pradesh                      Y.S.R.              Andhra Pradesh                           Y.S.R. (Kadapa)
1   state+bracket alias               Chhattisgarh         Uttar Bastar Kanker                Chhattisgarh              Uttar Bastar Kanker (Kanker)
2   state+bracket alias                    Haryana                       Mewat                     Haryana                               Nuh (Mewat)
3   state+bracket alias            Jammu & Kashmir                      Badgam             Jammu & Kashmir                           Ba

,District,State,NFHS-5 2019-21 MPI,Population living in households that use an improved sanitation facility2 %,Women age 15-24 years who use hygienic methods of protection during their menstrual period5 %,Children age 12-23 months fully vaccinated based on information from vaccination card only12 %,"Total children age 6-23 months receiving an adequate diet16, 17 %",All women age 15-49 years who are anaemic22 %,Men age 15 years and above wih high or very high >140 mg/dl Blood sugar level or taking medicine to control blood sugar level23 %,Women age 15 years and above who use any kind of tobacco %,Men age 15 years and above who consume alcohol %
0,Nicobars,Andaman & Nicobar Islands,0.013,83.45,100.00,94.10,18.70,38.30,15.39,63.46,64.49
1,North & Middle Andaman,Andaman & Nicobar Islands,0.019,86.38,100.00,NaN,5.90,62.07,18.28,46.77,45.26
2,South Andamans,Andaman & Nicobar Islands,0.004,89.33,98.19,96.60,23.49,57.66,18.10,19.60,32.81
3,Anantapur,Andhra Pradesh,0.027,71.25,95.01,96.60,2.91,50.45,16.75,5.03,16.84
4,Chittoor,Andhra Pradesh,0.022,74.62,94.66,87.50,5.96,51.80,22.42,4.30,18.30
...,...,...,...,...,...,...,...,...,...,...,...
702,Purba Bardhaman,West Bengal,0.059,64.15,80.42,97.20,34.10,77.86,21.79,9.93,22.94
703,Purba Medinipur Medinipur East,West Bengal,0.051,69.78,78.42,74.30,20.93,70.39,24.04,12.93,14.80
704,Puruliya,West Bengal,0.117,29.24,72.68,87.02,24.60,76.70,20.71,26.70,17.81
705,South 24 Parganas,West Bengal,0.045,70.08,80.31,96.34,27.55,61.62,21.28,6.28,16.54


In [23]:
display(merged_df)

os.makedirs("data/clean", exist_ok=True)
merged_df.to_excel("data/clean/MPI vs Health Indicators.xlsx", index=False)

,District,State,NFHS-5 2019-21 MPI,Population living in households that use an improved sanitation facility2 %,Women age 15-24 years who use hygienic methods of protection during their menstrual period5 %,Children age 12-23 months fully vaccinated based on information from vaccination card only12 %,"Total children age 6-23 months receiving an adequate diet16, 17 %",All women age 15-49 years who are anaemic22 %,Men age 15 years and above wih high or very high >140 mg/dl Blood sugar level or taking medicine to control blood sugar level23 %,Women age 15 years and above who use any kind of tobacco %,Men age 15 years and above who consume alcohol %
0,Nicobars,Andaman & Nicobar Islands,0.013,83.45,100.00,94.10,18.70,38.30,15.39,63.46,64.49
1,North & Middle Andaman,Andaman & Nicobar Islands,0.019,86.38,100.00,NaN,5.90,62.07,18.28,46.77,45.26
2,South Andamans,Andaman & Nicobar Islands,0.004,89.33,98.19,96.60,23.49,57.66,18.10,19.60,32.81
3,Anantapur,Andhra Pradesh,0.027,71.25,95.01,96.60,2.91,50.45,16.75,5.03,16.84
4,Chittoor,Andhra Pradesh,0.022,74.62,94.66,87.50,5.96,51.80,22.42,4.30,18.30
...,...,...,...,...,...,...,...,...,...,...,...
702,Purba Bardhaman,West Bengal,0.059,64.15,80.42,97.20,34.10,77.86,21.79,9.93,22.94
703,Purba Medinipur Medinipur East,West Bengal,0.051,69.78,78.42,74.30,20.93,70.39,24.04,12.93,14.80
704,Puruliya,West Bengal,0.117,29.24,72.68,87.02,24.60,76.70,20.71,26.70,17.81
705,South 24 Parganas,West Bengal,0.045,70.08,80.31,96.34,27.55,61.62,21.28,6.28,16.54
